# Quick Overview <a class="anchor" id="top"></a>

*Click the badge below to try this tutorial interactively in your browser:*

[![Launch Binder](../images/custom_binder_logo.svg)](https://mybinder.org/v2/gh/QSD-Group/QSDsan-env/main?urlpath=git-pull%3Frepo%3Dhttps%253A%252F%252Fgithub.com%252FQSD-group%252FQSDsan%26urlpath%3Dlab%252Ftree%252FQSDsan%252Fdocs%252Fsource%252Ftutorials%26branch%3Dmain)

You can also run this tutorial in [Google Colab](https://colab.research.google.com). It takes a one-time setup per session: follow the [Colab instructions](https://qsdsan.readthedocs.io/en/latest/tutorials/index.html#run-in-colab).

- **Prepared by:**

    - [Yalin Li](https://github.com/yalinli2)

- **Learning objectives.** After this tutorial, you will be able to:

    - See how the main `QSDsan` classes (`Component`, `WasteStream`, `SanUnit`, `System`, `TEA`, `LCA`, `Process`) fit together
    - Preview the typical workflow from defining components to running an analysis
    - Decide which subsequent tutorials to read based on what you want to build

`QSDsan` is a Python platform for the quantitative sustainable design of sanitation and resource recovery systems. Built on `BioSTEAM`, it lets you assemble treatment trains as connected unit operations, compute techno-economic and life-cycle impact metrics, and explore uncertainty and sensitivity across designs.

This tutorial is a brief tour of the main pieces. Each class shown here has its own dedicated tutorial; pointers are at the end.

<!-- tutorial-setup-section -->

## Setup

Import `QSDsan` and confirm the installed version.

In [ ]:
import qsdsan as qs, exposan
print(f'This tutorial was made with qsdsan v{qs.__version__} and exposan v{exposan.__version__}')

## 1. The pieces of QSDsan <a class="anchor" id="s1"></a>

`QSDsan` groups its functionality into a small set of core classes:

![QSDsan UML diagram](../images/uml.png)

*Thermosteam, BioSTEAM, and BW2QSD are external packages that QSDsan leverages.*

| Class | What it represents |
| --- | --- |
| `Component` | A chemical or biological species, with thermodynamic and biological properties |
| `WasteStream` | A flow of components, with composition and thermodynamic state |
| `SanUnit` | A unit operation, with `_run`, `_design`, and `_cost` methods |
| `System` | A connected set of units representing a treatment train |
| `Process` | Rate expressions and stoichiometry for dynamic biokinetic models |
| `TEA` | Techno-economic analysis on a `System` |
| `LCA` | Life cycle assessment on a `System` |
| `Model` | Parameters and metrics for uncertainty and sensitivity analyses |

The rest of this tour shows each one briefly.

## 2. Components and Streams <a class="anchor" id="s2"></a>

A `Component` describes one species (a chemical, a microbial group, or an aggregate parameter such as COD). A `Components` registry defines the thermodynamic basis for the model and must be set as the active thermo before building streams. See the [Component](https://qsdsan.readthedocs.io/en/latest/tutorials/2_Component.html) tutorial for more information.

In [ ]:
cmps = qs.Components.load_default()
qs.set_thermo(cmps)
cmps.show()

Each component carries a rich set of properties:

In [ ]:
cmps.S_CH4.show(chemical_info=True)

`QSDsan` provides three stream classes in a hierarchy. The [Stream](https://biosteam.readthedocs.io/en/latest/API/thermosteam/Stream.html) class is from Thermosteam (BioSTEAM's thermodynamic engine), which represents a generic material flow. `SanStream` adds stream-level life cycle impact functionalit, and `WasteStream` adds wastewater-modeling functionality on top of `SanStream`. See this [comparison page](https://qsdsan.readthedocs.io/en/latest/api/major_classes/streams.html) and the [WasteStream](https://qsdsan.readthedocs.io/en/latest/tutorials/3_WasteStream.html) tutorial for more information.

A `WasteStream` holds component flows plus the thermodynamic state. Several convenience initializers are available; for example, a typical municipal influent can be built from a COD-based model:

In [ ]:
ww = qs.WasteStream.codbased_inf_model('ww', flow_tot=100)
ww.show()

## 3. Unit Operations <a class="anchor" id="s3"></a>

A `SanUnit` is a unit operation. `QSDsan` have many pre-built subclasses (clarifiers, anaerobic digesters, membrane filters, etc.) that are often used for sanitation and resource recovery facilities. You can also subclass `SanUnit` to model your own; see the [basic](https://qsdsan.readthedocs.io/en/latest/tutorials/4_SanUnit_basic.html) and [advanced](https://qsdsan.readthedocs.io/en/latest/tutorials/5_SanUnit_advanced.html) tutorials for more information.

In [ ]:
# A small sample of built-in unit operations
# See the list of unit operations in the documentation for more details:
# https://qsdsan.readthedocs.io/en/latest/api/unit_operations/index.html
ops = sorted(i for i in dir(qs.unit_operations) if not i.startswith('_'))
print(f'qsdsan ships {len(ops)} unit operations. A few examples:')
for o in ops[:10]:
    print(f'  - {o}')

## 4. Process Models <a class="anchor" id="s4"></a>

A `Process` represents a single reaction defined by a rate expression and component stoichiometry. A `Processes` instance groups related `Process` objects into a complete dynamic model such as ASM1, ASM2d, or ADM1. See the [Process](https://qsdsan.readthedocs.io/en/latest/tutorials/10_Process.html) tutorial for more information.

**Note.** If your model is steady-state, i.e., flows and unit behavior do not change with time, you do not need `Process`. It is only needed when you need to capture time-varying behavior. The dynamic-modeling tutorials in Part III cover this in depth.

In [ ]:
asm1_cmps = qs.process_models.create_asm1_cmps(set_thermo=True)
asm1 = qs.process_models.ASM1()
print(asm1.rate_equations)

## 5. Assembling Unit Operations into a System <a class="anchor" id="s5"></a>

Units connect via influent and effluent streams to form a `System`. Below is an example from `EXPOsan` (the companion package of established systems). The `bwaise` study models sanitation for Bwaise, an unplanned settlement in Kampala, Uganda. See the [System](https://qsdsan.readthedocs.io/en/latest/tutorials/6_System.html) tutorial for more information.

In [ ]:
from exposan import bwaise as bw
bw.load()
bw.sysA.diagram()

Once simulated, you can read per-unit results:

In [ ]:
print(bw.A2.results())

## 6. Techno-Economic Analysis and Life Cycle Assessment (TEA/LCA) <a class="anchor" id="s6"></a>

A `TEA` runs the techno-economics on a `System`; an `LCA` runs the impact assessment. Both attach to the same `System` and read its simulation results. See the [TEA](https://qsdsan.readthedocs.io/en/latest/tutorials/7_TEA.html) and [LCA](https://qsdsan.readthedocs.io/en/latest/tutorials/8_LCA.html) tutorials for more information.

In [ ]:
c = qs.currency
print('For sysA in the Bwaise study:\n')
for attr in ('NPV', 'EAC', 'CAPEX', 'AOC', 'sales', 'net_earnings'):
    uom = c if attr in ('NPV', 'CAPEX') else f'{c}/yr'
    print(f'  - {attr}: {getattr(bw.teaA, attr):,.0f} {uom}')

The `LCA` object summarizes impacts by category and contributor:

In [ ]:
bw.lcaA

## 7. Uncertainty and Sensitivity Analyses <a class="anchor" id="s7"></a>

To perform uncertainty and sensitiviy analysis, a `Model` object is built on a `System` to add uncertain distributions of the input parameters and define output metrics of interest. The `Model` object can be then used to run Monte Carlo (for uncertainty) and sensitivity analyses. See the [Uncertainty and Sensitivity Analyses](https://qsdsan.readthedocs.io/en/latest/tutorials/9_Uncertainty_and_Sensitivity_Analyses.html) for more information.

In [ ]:
bw.modelA  # shows the declared uncertainty parameters

`QSDsan` also includes a [stats](https://qsdsan.readthedocs.io/en/latest/api/statistics.html) module for statistical analysis (e.g., global sensitivity analysis), including and visualization helpers.

## Where to go next <a class="anchor" id="next"></a>

The tutorials are organized in three parts. You just finished **Part I**.

**Part II.** QSDsan's core classes. Each tutorial dives into one or a few of the classes you just saw.

- [2. Component](https://qsdsan.readthedocs.io/en/latest/tutorials/2_Component.html)
- [3. WasteStream](https://qsdsan.readthedocs.io/en/latest/tutorials/3_WasteStream.html)
- [4. SanUnit (basic)](https://qsdsan.readthedocs.io/en/latest/tutorials/4_SanUnit_basic.html)
- [5. SanUnit (advanced)](https://qsdsan.readthedocs.io/en/latest/tutorials/5_SanUnit_advanced.html)
- [6. System](https://qsdsan.readthedocs.io/en/latest/tutorials/6_System.html)
- [7. TEA](https://qsdsan.readthedocs.io/en/latest/tutorials/7_TEA.html)
- [8. LCA](https://qsdsan.readthedocs.io/en/latest/tutorials/8_LCA.html)
- [9. Uncertainty and Sensitivity Analyses](https://qsdsan.readthedocs.io/en/latest/tutorials/9_Uncertainty_and_Sensitivity_Analyses.html)
- [10. Process](https://qsdsan.readthedocs.io/en/latest/tutorials/10_Process.html)

**Part III.** Integrating process models. Build dynamic, multi-component models on top of QSDsan.

- [11. Dynamic Simulation](https://qsdsan.readthedocs.io/en/latest/tutorials/11_Dynamic_Simulation.html)
- [12. ADM1 Case Study](https://qsdsan.readthedocs.io/en/latest/tutorials/12_Anaerobic_Digestion_Model_No_1.html)
- [13. Process Modeling 101](https://qsdsan.readthedocs.io/en/latest/tutorials/13_Process_Modeling_101.html)

<!-- tutorial-nav-footer -->

---

<a href="#top">? Back to top</a>